# Diagnostic Qwen3-VL FP8 — batterie complete

> `Kernel -> Restart Kernel and Run All Cells`

Ce notebook execute **tous les tests** dans l'ordre pour identifier precisement pourquoi
la lib `kernels` refuse de trouver `finegrained-fp8` v4, malgre le symlink.

Chaque cellule est independante et log son resultat en clair.
On ne s'arrete PAS sur une erreur — on collecte tout, puis on decide.

## 0. Variables d'environnement AVANT tout import

Toutes les variables connues qui pourraient debloquer le cache kernels.
Ces variables DOIVENT etre posees avant `import transformers` / `import kernels`.

In [ ]:
import os

# --- Mode offline complet ---
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"

# --- Trust ---
os.environ["HF_KERNELS_TRUST_REMOTE_CODE"] = "1"
os.environ["TRUST_REMOTE_CODE"] = "1"

# --- Caches ---
os.environ["HF_HOME"] = os.path.expanduser("~/.cache/huggingface")
os.environ["HF_KERNELS_CACHE"] = os.path.expanduser("~/.cache/huggingface/kernels")
os.environ["HUGGINGFACE_HUB_CACHE"] = os.path.expanduser("~/.cache/huggingface/hub")

# --- Desactivation kernels dynamiques (fallback PyTorch pur) ---
os.environ["DISABLE_KERNEL_MAPPING"] = "1"
os.environ["USE_KERNELS"] = "0"

print("Env variables posees :")
for k in sorted(os.environ):
    if any(x in k for x in ("HF_", "HUGGING", "KERNEL", "TRANSFORMERS", "TRUST")):
        print(f"  {k} = {os.environ[k]}")

## 1. Inventaire du systeme

In [ ]:
import sys, platform, subprocess
print("Python       :", sys.version.split()[0])
print("Platform     :", platform.platform())
print()
print("--- Packages critiques ---")
for pkg in ["transformers", "kernels", "torch", "accelerate", "huggingface_hub", "compressed_tensors"]:
    try:
        mod = __import__(pkg)
        v = getattr(mod, "__version__", "?")
        f = getattr(mod, "__file__", "?")
        print(f"  {pkg:22s} {v:12s}  {f}")
    except ImportError as e:
        print(f"  {pkg:22s} NON INSTALLE ({e})")

In [ ]:
import torch
print("CUDA available :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA version   :", torch.version.cuda)
    print("Device         :", torch.cuda.get_device_name(0))
    print("Capability     :", torch.cuda.get_device_capability(0))
    print("VRAM total     :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

## 2. Ou est physiquement le kernel v4 sur le disque

In [ ]:
import subprocess

print("--- find /domino pour finegrained-fp8 ---")
r = subprocess.run(
    ["find", "/domino", "-type", "d", "-name", "finegrained-fp8"],
    capture_output=True, text=True, timeout=60
)
print(r.stdout or "(rien)")
if r.stderr: print("stderr:", r.stderr[:500])

In [ ]:
import subprocess

print("--- Arborescence de finegrained-fp8 sur Domino ---")
r = subprocess.run(
    ["find", "/domino/edv/modelhub/ModelHub-model-huggingface-kernels-community/finegrained-fp8",
     "-maxdepth", "4", "-type", "d"],
    capture_output=True, text=True, timeout=60
)
print(r.stdout or "(rien)")

In [ ]:
import subprocess, os

print("--- Symlink actuel dans ~/.cache/huggingface/kernels/ ---")
r = subprocess.run(
    ["ls", "-la", os.path.expanduser("~/.cache/huggingface/kernels/kernels-community/")],
    capture_output=True, text=True
)
print(r.stdout)
if r.stderr: print("stderr:", r.stderr)

print("\n--- Contenu du symlink finegrained-fp8/ ---")
r = subprocess.run(
    ["ls", "-la", os.path.expanduser("~/.cache/huggingface/kernels/kernels-community/finegrained-fp8/")],
    capture_output=True, text=True
)
print(r.stdout)

print("\n--- Contenu de v4/ (build/, reports/, README.md attendus) ---")
r = subprocess.run(
    ["ls", "-la", os.path.expanduser("~/.cache/huggingface/kernels/kernels-community/finegrained-fp8/build/")],
    capture_output=True, text=True
)
print(r.stdout or "(pas de build/ — le symlink pointe peut-etre au mauvais endroit)")

## 3. Ou la lib `kernels` cherche-t-elle reellement ?

C'est LE point cle. Le symlink est peut-etre au bon endroit selon nous,
mais la lib peut chercher ailleurs (autre variable d'env, autre convention de path).

In [ ]:
try:
    import kernels
    import inspect
    print("kernels module     :", kernels.__file__)
    print("kernels version    :", getattr(kernels, "__version__", "?"))
    print()

    # Chercher les fonctions de resolution de cache
    print("--- Fonctions/attributs interessants ---")
    for name in dir(kernels):
        if not name.startswith("_") and any(x in name.lower() for x in ("cache", "path", "dir", "get_kernel", "load")):
            obj = getattr(kernels, name)
            print(f"  {name} -> {type(obj).__name__}")
except Exception as e:
    print("ERREUR import kernels:", e)

In [ ]:
# Inspecter le module kernels pour trouver ou il resout le cache
try:
    import kernels, os
    kernels_dir = os.path.dirname(kernels.__file__)
    print("Fichiers dans le module kernels:")
    for f in sorted(os.listdir(kernels_dir)):
        if f.endswith(".py"):
            print(f"  {f}")
    print()

    # Chercher les references a un cache dir dans le code source
    import subprocess
    print("--- grep 'cache' dans le code de kernels/ ---")
    r = subprocess.run(
        ["grep", "-rn", "-E", "(cache|CACHE|HF_KERNELS|HOME)",
         "--include=*.py", kernels_dir],
        capture_output=True, text=True
    )
    # Limiter aux 40 premieres lignes pertinentes
    lines = [l for l in r.stdout.split("\n") if l and "test" not in l.lower()][:40]
    print("\n".join(lines))
except Exception as e:
    print("ERREUR:", e)

In [ ]:
# Test frontal: essayer get_kernel et lire l'erreur EXACTE (avec traceback)
import traceback
try:
    from kernels import get_kernel
    k = get_kernel("kernels-community/finegrained-fp8")
    print("SUCCES:", k)
except Exception as e:
    print("ECHEC — type:", type(e).__name__)
    print("Message:", e)
    print()
    print("--- Traceback complet ---")
    traceback.print_exc()

In [ ]:
# Meme test mais en essayant plusieurs revisions
import traceback
try:
    from kernels import get_kernel
    for rev in ["v4", "main", "4", None]:
        print(f"\n--- Essai avec revision={rev!r} ---")
        try:
            if rev is None:
                k = get_kernel("kernels-community/finegrained-fp8")
            else:
                k = get_kernel("kernels-community/finegrained-fp8", revision=rev)
            print(f"  SUCCES: {k}")
            break
        except Exception as e:
            print(f"  ECHEC ({type(e).__name__}): {str(e)[:200]}")
except Exception as e:
    print("Import failed:", e)

## 4. Structure attendue par `kernels` vs structure reelle

La lib `kernels` a probablement une convention precise :
`<cache>/<namespace>/<repo>/<version_or_commit>/build/...`

On liste tout ce qui existe pour verifier.

In [ ]:
import os, subprocess

paths_to_check = [
    "~/.cache/huggingface/kernels",
    "~/.cache/huggingface/hub",
    "~/.cache/kernels",
    "/opt/conda/envs/DWS-GPU/lib/python3.11/site-packages/kernels",
    "/domino/edv/modelhub/ModelHub-model-huggingface-kernels-community",
]

for p in paths_to_check:
    p_expanded = os.path.expanduser(p)
    print(f"\n=== {p_expanded} ===")
    if os.path.exists(p_expanded):
        r = subprocess.run(["find", p_expanded, "-maxdepth", "5", "-type", "d"],
                           capture_output=True, text=True, timeout=30)
        for line in r.stdout.split("\n")[:30]:
            print(" ", line)
    else:
        print("  (n'existe pas)")

## 5. Tester le chargement du modele avec differentes strategies

In [ ]:
# Verifier d'abord que le chemin modele existe
import os
from pathlib import Path

MODEL_PATH = "/domino/edv/modelhub/ModelHub-model-huggingface-Qwen/Qwen3.6-VL-7B-FP8/main"

print(f"MODEL_PATH = {MODEL_PATH}")
print(f"  existe          : {os.path.isdir(MODEL_PATH)}")
print(f"  config.json     : {(Path(MODEL_PATH) / 'config.json').exists()}")

if not os.path.isdir(MODEL_PATH):
    # Chercher le vrai chemin
    import subprocess
    print("\n--- Recherche du vrai chemin modele ---")
    r = subprocess.run(
        ["find", "/domino/edv/modelhub", "-name", "config.json", "-path", "*Qwen*VL*"],
        capture_output=True, text=True, timeout=60
    )
    print(r.stdout or "(rien trouve)")

In [ ]:
# Strategie 1: from_pretrained avec kernels desactives
import time, torch
from transformers import AutoProcessor, AutoModelForImageTextToText

print("=== Strategie 1: use_kernels=False + DISABLE_KERNEL_MAPPING=1 ===")
t0 = time.time()
try:
    processor = AutoProcessor.from_pretrained(
        MODEL_PATH,
        trust_remote_code=True,
        local_files_only=True,
    )
    processor.tokenizer.padding_side = "left"
    print(f"  Processor OK en {time.time()-t0:.1f}s")

    t0 = time.time()
    # Essayer avec use_kernels si supporte
    kwargs = dict(
        dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True,
        low_cpu_mem_usage=True,
        local_files_only=True,
    )
    # Detecter si use_kernels est supporte
    import inspect
    sig = inspect.signature(AutoModelForImageTextToText.from_pretrained)
    if "use_kernels" in sig.parameters:
        kwargs["use_kernels"] = False
        print("  (use_kernels=False ajoute — supporte par cette version)")
    else:
        print("  (use_kernels PAS supporte dans from_pretrained — on compte sur DISABLE_KERNEL_MAPPING)")

    model = AutoModelForImageTextToText.from_pretrained(MODEL_PATH, **kwargs)
    model.eval()
    print(f"  Modele charge en {time.time()-t0:.1f}s")
    print(f"  dtype: {next(model.parameters()).dtype}")
    print(f"  device: {next(model.parameters()).device}")
    LOADED = True
except Exception as e:
    print(f"  ECHEC: {type(e).__name__}: {e}")
    import traceback; traceback.print_exc()
    LOADED = False

In [ ]:
# Test inference sur une image factice (juste pour verifier que le forward passe)
if LOADED:
    print("=== Test forward pass ===")
    from PIL import Image
    import numpy as np, torch, time

    img = Image.fromarray((np.ones((512, 512, 3), dtype=np.uint8) * 200))
    messages = [{"role": "user", "content": [
        {"type": "image", "image": img},
        {"type": "text", "text": "Que vois-tu ?"},
    ]}]

    try:
        text_in = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    except Exception:
        text_in = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    print(f"  Chat template OK, len={len(text_in)}")

    try:
        inputs = processor(text=[text_in], images=[img], return_tensors="pt").to(model.device)
        print(f"  Processor OK, input_ids shape: {inputs['input_ids'].shape}")

        t0 = time.time()
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=50, do_sample=False,
                                 pad_token_id=processor.tokenizer.eos_token_id)
        elapsed = time.time() - t0
        generated = out[0][inputs["input_ids"].shape[1]:]
        text = processor.decode(generated, skip_special_tokens=True)
        print(f"  Generation OK en {elapsed:.1f}s")
        print(f"  Reponse: {text[:200]}")
    except Exception as e:
        print(f"  ECHEC forward: {type(e).__name__}: {e}")
        import traceback; traceback.print_exc()
else:
    print("Modele pas charge, on saute le forward.")

## 6. Bilan et recommandation

Cette derniere cellule synthetise les resultats pour decider de la suite.

In [ ]:
print("=" * 60)
print("BILAN")
print("=" * 60)

# 1. Env
print("\n1. Env variables offline/trust posees:", "HF_HUB_OFFLINE" in os.environ)

# 2. Symlink
sl = os.path.expanduser("~/.cache/huggingface/kernels/kernels-community/finegrained-fp8")
sl_ok = os.path.islink(sl) or os.path.isdir(sl)
sl_has_build = os.path.isdir(os.path.join(sl, "build"))
print(f"2. Symlink existe: {sl_ok}, contient build/: {sl_has_build}")

# 3. get_kernel
try:
    from kernels import get_kernel
    get_kernel("kernels-community/finegrained-fp8")
    print("3. get_kernel: OK")
    gk = True
except Exception as e:
    print(f"3. get_kernel: ECHEC ({type(e).__name__}: {str(e)[:80]})")
    gk = False

# 4. Modele
print(f"4. Modele charge: {LOADED if 'LOADED' in dir() else '(pas teste)'}")

print()
print("=" * 60)
print("RECOMMANDATION")
print("=" * 60)
if gk:
    print("get_kernel marche -> le probleme initial est resolu, tu peux lancer le pipeline OCR complet.")
elif LOADED:
    print("get_kernel echoue MAIS le modele charge quand meme grace au fallback.")
    print("-> Lance le pipeline: il tournera en bf16 pur (un peu plus lent mais OK).")
else:
    print("Ni les kernels ni le modele ne chargent.")
    print("-> Regarde les tracebacks des cellules 3 et 5 pour identifier la vraie cause.")
    print("-> En dernier recours: passer sur le checkpoint BF16 non-FP8 du meme modele.")